# 🌊 Agentic AI Flood Risk Detection System
# An autonomous AI agent for flood risk detection using LangGraph, Gemini, and scikit-learn
# Author: Md Rasel Uddin | Tulane University RCSE

In [2]:
!pip install -q langgraph langchain langchain-google-genai google-generativeai requests joblib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 3.5 MB/s eta 0:00:00


In [3]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")
os.environ["TELEGRAM_BOT_TOKEN"] = getpass("Telegram Bot Token: ")
os.environ["TELEGRAM_CHAT_ID"] = getpass("Telegram Chat ID: ")
os.environ["OPENWEATHER_API_KEY"] = getpass("OpenWeatherMap Key: ")

Gemini API Key: ··········
Telegram Bot Token: ··········
Telegram Chat ID: ··········
OpenWeatherMap Key: ··········


In [4]:
from google.colab import files
import joblib

print("👉 model.pkl ফাইল upload করুন:")
model_upload = files.upload()
MODEL_PATH = list(model_upload.keys())[0]

print("👉 scaler.pkl ফাইল upload করুন:")
scaler_upload = files.upload()
SCALER_PATH = list(scaler_upload.keys())[0]

rf_model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print("✅ Model & Scaler loaded successfully!")

👉 model.pkl ফাইল upload করুন:


Saving flood_model.pkl to flood_model.pkl
👉 scaler.pkl ফাইল upload করুন:


Saving scaler.pkl to scaler.pkl
✅ Model & Scaler loaded successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.1. This might lead to b

In [5]:
import requests

def get_weather_data(city: str = "Dhaka"):
    """OpenWeatherMap থেকে বর্তমান আবহাওয়া তথ্য আনে"""
    api_key = os.environ["OPENWEATHER_API_KEY"]
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url)
    data = response.json()
    if response.status_code != 200:
        return {"error": data.get("message", "Unknown error")}
    return {
        "city": city,
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "rainfall_last_1h": data.get("rain", {}).get("1h", 0),
        "weather_condition": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"]
    }

def predict_flood_risk(rainfall: float, river_level: float, temperature: float):
    """Random Forest model দিয়ে flood risk predict করে"""
    input_data = [[rainfall, river_level, temperature]]
    input_scaled = scaler.transform(input_data)
    prediction = rf_model.predict(input_scaled)[0]
    probability = rf_model.predict_proba(input_scaled)[0]
    risk_level = "High Risk" if prediction == 1 else "Low Risk"
    risk_percentage = int(probability[1] * 100)
    return {
        "risk_level": risk_level,
        "risk_percentage": risk_percentage,
        "rainfall": rainfall,
        "river_level": river_level,
        "temperature": temperature
    }

def send_telegram_alert(message: str):
    """Telegram bot দিয়ে সতর্কবার্তা পাঠায়"""
    token = os.environ["TELEGRAM_BOT_TOKEN"]
    chat_id = os.environ["TELEGRAM_CHAT_ID"]
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message, "parse_mode": "Markdown"}
    response = requests.post(url, data=payload)
    return response.json()

In [6]:
from langchain_core.tools import tool

@tool
def fetch_weather(city: str) -> dict:
    """একটি শহরের বর্তমান আবহাওয়া, বৃষ্টিপাত ও তাপমাত্রা নিয়ে আসে। Flood risk analyze করার আগে এটা ব্যবহার করো।"""
    return get_weather_data(city)

@tool
def check_flood_risk(rainfall: float, river_level: float, temperature: float) -> dict:
    """Random Forest ML model ব্যবহার করে flood risk predict করে। rainfall (mm), river_level (মিটার), temperature (°C) দরকার।"""
    return predict_flood_risk(rainfall, river_level, temperature)

@tool
def send_alert(message: str) -> str:
    """High flood risk পাওয়া গেলে Telegram এ জরুরি সতর্কবার্তা পাঠায়।"""
    send_telegram_alert(message)
    return "Alert sent successfully"

tools = [fetch_weather, check_flood_risk, send_alert]

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

system_prompt = """তুমি একজন Flood Risk Monitoring Agent। তোমার কাজ:

1. যে এলাকার নাম দেওয়া হবে, সেই এলাকার আবহাওয়া তথ্য (fetch_weather) নিয়ে আসো
2. সেই তথ্য দিয়ে flood risk check করো (check_flood_risk) — river_level না জানা থাকলে rainfall এর উপর ভিত্তি করে যুক্তিসঙ্গত মান অনুমান করো
3. risk "High Risk" বা risk_percentage 50+ হলে send_alert দিয়ে বাংলায় actionable সতর্কবার্তা পাঠাও
4. risk কম হলে শুধু সংক্ষিপ্ত status জানাও

সবসময় শেষে বাংলায় সংক্ষিপ্ত সারমর্ম দাও।
"""

agent = create_react_agent(llm, tools, prompt=system_prompt)
print("✅ Agent ready!")

✅ Agent ready!


/tmp/ipykernel_4304/575104844.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


In [8]:
response = agent.invoke({
    "messages": [("user", "Dhaka এলাকার flood risk check করো এবং প্রয়োজন হলে alert পাঠাও")]
})

print(response["messages"][-1].content)

Dhaka এলাকার জন্য flood risk "Low Risk" এবং risk percentage 28%। কোনো alert পাঠানোর প্রয়োজন নেই।
